# 原 Stage1 39项输入：现金与 SOXL 相对选择双接收任务

复用冻结现金头，只新增一项相对头；两项都需通过固定信息门。来源、预算与边界见 plan.json。

In [1]:
import hashlib
import json
import os
import shlex
import subprocess
import tarfile
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _ssh, archive_nas_recorder

os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path.cwd().resolve();card=root/'research/stage1_dual_task_39_information_20261023'
plan=json.loads((card/'plan.json').read_text());source=plan['source']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and not (card/'run_state.json').exists()
assert plan['budget']=={'new_market_requests':0,'source_Recorder_object_loads_max':6,'DatasetH_builds':1,'new_model_fit_attempts_max':1,'bootstrap_attempts_max':2,'new_writable_Qlib_Recorders_max':1,'native_accounts':0}
assert sha(root/source['provider_uri']/'qlib_store_manifest.json')==source['provider_manifest_sha256']
assert sha(root/'src/analysis/seed_disagreement_information.py')==source['bootstrap_module_sha256']
parent=root/source['parent_card'];parent_summary=json.loads((parent/'summary.json').read_text());parent_manifest=json.loads((parent/'verification_manifest.json').read_text())
assert sha(parent/'summary.json')==source['parent_summary_sha256'] and sha(parent/'verification_manifest.json')==source['parent_manifest_sha256']
assert parent_summary['recorder_id']==source['parent_recorder_id'] and parent_summary['nas_evidence']['sha256']==source['parent_NAS_archive_sha256']
assert json.loads((parent/'nas_binding.json').read_text())['evidence']['sha256']==source['parent_NAS_archive_sha256']
for artifact,key in [('inputs/features','parent_features_sha256'),('inputs/daily_frame','parent_frame_sha256'),('inputs/timing','parent_timing_sha256'),('models/base_outer_71','parent_model_sha256')]:
    matches=[parent/e['path'] for e in parent_manifest['entries'] if e['path'].endswith('/'+artifact)]
    assert len(matches)==1 and sha(matches[0])==source[key],artifact
cash_card=root/source['cash_head_card'];cash_summary=json.loads((cash_card/'summary.json').read_text());cash_binding=json.loads((cash_card/'nas_binding.json').read_text())
assert sha(cash_card/'summary.json')==source['cash_head_summary_sha256']
assert cash_summary['recorder_id']==source['cash_head_recorder_id'] and cash_summary['nas_evidence']['sha256']==cash_binding['evidence']['sha256']==source['cash_head_NAS_archive_sha256']
cash_local=cash_card/'qlib_recorder'/cash_summary['experiment_id']/cash_summary['recorder_id']/'artifacts'
for artifact,key,szkey in [('models/baseline','cash_head_model_sha256','cash_head_model_bytes'),('predictions/baseline','cash_head_predictions_sha256','cash_head_predictions_bytes')]:
    p=cash_local/artifact;assert sha(p)==source[key] and p.stat().st_size==source[szkey]
with tempfile.TemporaryDirectory(prefix='stage1-dual-cash-source-') as tmp:
    archive=Path(tmp)/'source.tar.gz'
    with archive.open('wb') as out:subprocess.run(['ssh','-o','BatchMode=yes','-p',str(cash_binding['ssh']['port']),cash_binding['ssh']['host'],'cat',cash_summary['nas_evidence']['nas_path']],stdout=out,stderr=subprocess.PIPE,check=True)
    assert sha(archive)==source['cash_head_NAS_archive_sha256']
    with tarfile.open(archive,'r:gz') as tar:
        names=set(tar.getnames());restored=[]
        for artifact,key,szkey in [('models/baseline','cash_head_model_sha256','cash_head_model_bytes'),('predictions/baseline','cash_head_predictions_sha256','cash_head_predictions_bytes')]:
            name='qlib_recorder/'+cash_summary['experiment_id']+'/'+cash_summary['recorder_id']+'/artifacts/'+artifact
            assert name in names
            raw=tar.extractfile(name).read();assert len(raw)==source[szkey] and hashlib.sha256(raw).hexdigest()==source[key]
            restored.append(artifact)
atomic_json({'status':'verified_before_new_fit','NAS_archive_sha256':source['cash_head_NAS_archive_sha256'],'scoped_objects':restored,'all_sha256_and_sizes_match':True},card/'source_restore.json')
qlib.init(provider_uri=str(root/source['provider_uri']),region='us')
assert len(D.calendar(start_time='2011-01-03',end_time='2026-10-07',freq='day'))>=3964
with R.uri_context(str(parent/'qlib_recorder')):
    r=R.get_recorder(experiment_name=parent.name,recorder_id=source['parent_recorder_id'])
    original=r.load_object('models/base_outer_71');features=r.load_object('inputs/features');frame=r.load_object('inputs/daily_frame');timing=r.load_object('inputs/timing')
with R.uri_context(str(cash_card/'qlib_recorder')):
    r=R.get_recorder(experiment_name=cash_card.name,recorder_id=source['cash_head_recorder_id'])
    cash_model=r.load_object('models/baseline');cash_saved=r.load_object('predictions/baseline')
base_names=list(original.base_features)
assert len(base_names)==39 and set(base_names).issubset(features.columns)
assert frame.index.equals(features.index) and frame.index.equals(timing.index)
def h5(series):
    parts=[series.astype(float).shift(-i) for i in range(5)]
    valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
    return np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid)
soxx_h5=h5(frame.soxx_forward_open_return)
soxl_h5=h5(frame.soxl_forward_open_return)
cash_target=-soxx_h5
relative_target=soxl_h5-soxx_h5
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
train_cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC')
held_terminal=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train=(frame.index>=pd.Timestamp('2011-01-03'))&(frame.index<=pd.Timestamp('2022-12-30'))&(mature<train_cutoff)&cash_target.notna()&relative_target.notna()
held=(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))&(mature<held_terminal)&cash_target.notna()&relative_target.notna()
assert int(train.sum())==plan['cohorts']['expected_train_mature_rows'] and int(held.sum())==plan['cohorts']['expected_held_mature_rows']
assert len(cash_saved)==940 and cash_saved.signal_date.equals(pd.Series(frame.index[held],name='signal_date').reset_index(drop=True))
np.testing.assert_allclose(cash_saved.target.to_numpy(float),soxx_h5.loc[held].to_numpy(float),atol=1e-12,rtol=0)
static=pd.concat({'feature':features[base_names],'label':pd.DataFrame({'cash_advantage_H5':cash_target,'SOXL_relative_H5':relative_target}),'meta':pd.DataFrame({'label_available_at':mature,'train_mature':train,'held_mature':held})},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused_diagnostic'])})
tr=dataset.prepare('train',col_set=['feature','label','meta']);he=dataset.prepare('held',col_set=['feature','label','meta'])
tr=tr.loc[tr[('meta','train_mature')].astype(bool)];he=he.loc[he[('meta','held_mature')].astype(bool)]
assert len(tr)==3014 and len(he)==940
xtr=tr['feature'][base_names].to_numpy(float);xhe=he['feature'][base_names].to_numpy(float)
y_cash=he[('label','cash_advantage_H5')].to_numpy(float);y_rel=he[('label','SOXL_relative_H5')].to_numpy(float)
assert np.isfinite(y_cash).all() and np.isfinite(y_rel).all()
np.testing.assert_allclose(-cash_saved.target.to_numpy(float),y_cash,atol=1e-12,rtol=0)
cash_prediction=-cash_saved.prediction.to_numpy(float)
np.testing.assert_allclose(cash_model.predict(xhe),cash_saved.prediction.to_numpy(float),atol=1e-10,rtol=0)
mean_cash=float(tr[('label','cash_advantage_H5')].mean());mean_rel=float(tr[('label','SOXL_relative_H5')].mean())
preflight={'status':'two_original39_receiver_labels_calendar_and_reused_cash_head_verified','plan_sha256':sha(card/'plan.json'),'parent_rows':len(frame),'train_mature_rows':len(tr),'held_mature_rows':len(he),'source_Recorder_loads':6,'DatasetH_builds':1,'reused_cash_head_model_and_predictions_sha256_verified':True,'reused_cash_head_NAS_scoped_restore_verified':True,'new_model_fits':0,'new_accounts':0}
atomic_json(preflight,card/'preflight_result.json')
state={'status':'relative_head_fit_attempted','plan_sha256':sha(card/'plan.json'),'new_fit_attempts':1,'new_fits_completed':0,'bootstrap_attempts':0,'new_native_accounts':0}
atomic_json(state,card/'run_state.json')
relative_model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
relative_model.fit(xtr,tr[('label','SOXL_relative_H5')].to_numpy(float))
relative_prediction=relative_model.predict(xhe)
assert np.isfinite(relative_prediction).all()
state['new_fits_completed']=1;atomic_json(state,card/'run_state.json')
dates=pd.DatetimeIndex(he.index.get_level_values('datetime'))
scored=pd.DataFrame({'signal_date':dates,'cash_target':y_cash,'cash_prediction':cash_prediction,'cash_control':mean_cash,'SOXL_relative_target':y_rel,'SOXL_relative_prediction':relative_prediction,'SOXL_relative_control':mean_rel})
for key in ('cash','SOXL_relative'):
    scored[key+'_paired_mse_improvement']=(scored[key+'_target']-scored[key+'_control'])**2-(scored[key+'_target']-scored[key+'_prediction'])**2
full=pd.DataFrame({'signal_date':frame.index[(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))]})
full=full.merge(scored,on='signal_date',how='left',validate='one_to_one')
assert len(full)==944 and scored.signal_date.is_monotonic_increasing and int(full.cash_paired_mse_improvement.notna().sum())==940
boot=plan['joint_information_gate']['bootstrap'];metrics={};gates={};replicates={};indices=None
for key in ('cash','SOXL_relative'):
    diff=scored[key+'_paired_mse_improvement']
    half=[]
    for start,end in plan['cohorts']['held_halves']:
        g=scored.loc[scored.signal_date.between(start,end)]
        half.append({'start':start,'end':end,'rows':len(g),'paired_mean_mse_improvement':float(g[key+'_paired_mse_improvement'].mean())})
    annual=[]
    for year in plan['cohorts']['complete_held_years']:
        g=scored.loc[scored.signal_date.dt.year.eq(year)]
        annual.append({'year':year,'rows':len(g),'paired_mean_mse_improvement':float(g[key+'_paired_mse_improvement'].mean())})
    state['bootstrap_attempts']+=1;atomic_json(state,card/'run_state.json')
    interval,idx,rep=full_calendar_bootstrap(full[key+'_paired_mse_improvement'].to_numpy(float),mean_block=boot['mean_block_sessions'],resamples=boot['draws_each'],seed=boot['seed'],alpha=boot['one_sided_alpha_each'])
    assert interval['valid_replicates']==boot['draws_each']
    if indices is None:indices=idx
    else:assert np.array_equal(indices,idx)
    replicates[key]=rep
    metrics[key]={'control_train_mean':mean_cash if key=='cash' else mean_rel,'control_held_mse':float(np.mean((scored[key+'_target']-scored[key+'_control'])**2)),'model_held_mse':float(np.mean((scored[key+'_target']-scored[key+'_prediction'])**2)),'paired_mean_mse_improvement':float(diff.mean()),'halves':half,'complete_years':annual,'bootstrap':interval}
    gates[key]={'full_held_positive':metrics[key]['paired_mean_mse_improvement']>0,'both_halves_positive':all(x['paired_mean_mse_improvement']>0 for x in half),'both_complete_years_positive':all(x['paired_mean_mse_improvement']>0 for x in annual),'corrected_lower_positive':interval['lower_bound']>0}
joint_pass=all(all(x.values()) for x in gates.values())
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
    recorder=R.get_recorder();dataset.config(dump_all=True,recursive=True)
    recorder.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,dataset=dataset,source_model_hashes={'cash_reused':source['cash_head_model_sha256'],'relative_new':'one_fit'})
    recorder.save_objects(artifact_path='models',SOXL_relative=relative_model)
    recorder.save_objects(artifact_path='predictions',scored=scored)
    recorder.save_objects(artifact_path='analysis',full_held_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=replicates,bootstrap_indices=indices,joint_information_pass=joint_pass)
    recorder.log_metrics(new_model_fits=1,new_native_accounts=0,cash_improvement=metrics['cash']['paired_mean_mse_improvement'],SOXL_relative_improvement=metrics['SOXL_relative']['paired_mean_mse_improvement'],joint_information_pass=int(joint_pass))
    recorder_id,experiment_id=recorder.id,str(recorder.experiment_id)
summary={'experiment':plan['experiment'],'decision':'conditional_original39_dual_receiver_information_pass_needs_frozen_0bp_account_mapping' if joint_pass else 'not_promoted_fixed_original39_dual_receiver_information_failed','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'preflight':preflight,'joint_information_pass':joint_pass,'metrics':metrics,'gates':gates,'new_model_fits':1,'reused_cash_head_fits':1,'new_bootstraps':2,'new_native_accounts':0,'held_market_history_reused':True,'independent_forward_qualified_days':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']},'limitations':['Price H5 information labels omit raw-share dividend and Tiered funded-account effects.','2023-2026 history and saved cash head have been reused for earlier research; no independent forward qualification.','A positive model-versus-prior-mean screen would not prove incremental economic value versus frozen Stage1 weights or yield a predeclared position mapping.']}
atomic_json(summary,card/'summary.json')
binding=json.loads((parent/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2026-10-09')
atomic_json(binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':remote,'bytes':evidence['size_bytes'],'verified':True}
atomic_json(summary,card/'summary.json')
state.update(status='completed',recorder_id=recorder_id,experiment_id=experiment_id)
atomic_json(state,card/'run_state.json')
(card/'conclusion.md').write_text(
    '# 原 Stage1 39项输入：现金与SOXL相对选择双接收任务\n\n'
    f'固定2011–2022训练成熟3014日、复用2023–2026评价成熟940日。现金H5头按精确封存的原39输入SOXX预测反号复用，新增拟合仅一项SOXL−SOXX H5相对头；两头均以各自训练均值为无技能控制，评价全944日保留末端4个未知。0经济账户。\n\n'
    f'现金头相对训练均值的留出MSE改善{metrics["cash"]["paired_mean_mse_improvement"]:+.9g}，校正区块下界{metrics["cash"]["bootstrap"]["lower_bound"]:+.9g}；SOXL相对头MSE改善{metrics["SOXL_relative"]["paired_mean_mse_improvement"]:+.9g}，下界{metrics["SOXL_relative"]["bootstrap"]["lower_bound"]:+.9g}。两半段及2024/2025完整年度逐头门见summary.json，联合通过={joint_pass}。\n\n'
    f'决定`{summary["decision"]}`。价格标签不是带现金、原股分红与Tiered费用的实际账户；两头分别预测若无可靠增量，不得把预测强行变仓位。全部日期、模型、预测、两套成对bootstrap存在原生Qlib Recorder `{recorder_id}`，NAS SHA `{remote}`。历史已复用，SOXX 0bp/2bp/5bp全账户、30%回撤、CVaR、年度/滚动及独立前向均未由此取得。\n'
)
print(summary['decision'],metrics,gates,recorder_id)


not_promoted_fixed_original39_dual_receiver_information_failed {'cash': {'control_train_mean': -0.0032255746731863185, 'control_held_mse': 0.0023466841316487535, 'model_held_mse': 0.0025262669365982205, 'paired_mean_mse_improvement': -0.00017958280494946737, 'halves': [{'start': '2023-01-03', 'end': '2024-12-31', 'rows': 502, 'paired_mean_mse_improvement': -7.603950759173151e-05}, {'start': '2025-01-02', 'end': '2026-10-07', 'rows': 438, 'paired_mean_mse_improvement': -0.0002982557165329912}], 'complete_years': [{'year': 2024, 'rows': 252, 'paired_mean_mse_improvement': 4.860755535483653e-06}, {'year': 2025, 'rows': 250, 'paired_mean_mse_improvement': -0.0002145025684821343}], 'bootstrap': {'paired_mean_loss_improvement': -0.0001795828049494674, 'lower_bound': -0.0004454941692562761, 'upper_bound': 6.512196527160112e-05, 'valid_replicates': 10000, 'invalid_replicates': 0, 'calendar_rows': 944, 'scored_rows': 940}}, 'SOXL_relative': {'control_train_mean': 0.0008179531594095632, 'control

## 中文结论

固定原39项Ridge双接收任务失败：现金头相对训练均值MSE改善−0.000179583、SOXL−SOXX H5头−0.001237508，两个分段和两完整年度的联合门未通过，两校正区块下界均负。现金头复用既有封存模型，新增1次相对头拟合、2次固定bootstrap、0账户。现金源2工件及新Recorder15工件均经NAS限定恢复，Qlib读回通过。历史价格信息任务不能证明SOXX费用后超额；不把分数强行映射仓位。